Synthetic files

In [22]:
%%writefile generate_data.py
import argparse
import csv
import hashlib
import json
import math
import os
import random
from collections import defaultdict
from datetime import date, datetime, timedelta

DAILY_ROWS = {"arrivals": 207, "pos": 1500, "proc": 100, "erp": 2000, "wifi": 5000}

ROOM_TYPES = [
    ("Standard Twin", 550, 25),
    ("Deluxe King", 950, 35),
    ("Deluxe Twin", 850, 15),
    ("Executive Suite", 1800, 15),
    ("Family Room", 1200, 10),
]
CHANNELS = [("Direct", 35), ("OTA", 45), ("Corporate", 20)]
STAY_NIGHTS = [(1, 20), (2, 30), (3, 25), (4, 12), (5, 8), (7, 5)]

DEPARTMENTS = [
    "Rooms", "Front Office", "Housekeeping", "F&B", "Kitchen", "Banquets",
    "Engineering", "Spa & Wellness", "Recreation", "Security", "Concierge",
    "Sales & Marketing", "IT", "Finance", "Administration",
]
DEPT_VARIANTS = {
    "F&B": ["Food and Beverage", "F and B", "FnB"],
    "Housekeeping": ["HK", "House Keeping", "housekeeping"],
    "Front Office": ["Front Desk", "FO"],
    "Engineering": ["Eng", "Maintenance"],
    "Spa & Wellness": ["Spa", "Spa and Wellness"],
    "Sales & Marketing": ["Sales and Marketing", "S&M"],
    "Rooms": ["Rooms Division", "rooms"],
}

HR_BASE = {
    "Rooms": 30, "Front Office": 25, "Housekeeping": 60, "F&B": 55,
    "Kitchen": 40, "Banquets": 18, "Engineering": 22, "Spa & Wellness": 14,
    "Recreation": 10, "Security": 20, "Concierge": 8, "Sales & Marketing": 9,
    "IT": 7, "Finance": 8, "Administration": 8,
}

ERP_ACCOUNTS = [
    ("4100", "Rooms", "Credit", 3500, 12),
    ("4200", "F&B", "Credit", 900, 20),
    ("4300", "Spa & Wellness", "Credit", 600, 4),
    ("4400", "Banquets", "Credit", 8000, 2),
    ("5100", "F&B", "Debit", 700, 15),
    ("5200", "Housekeeping", "Debit", 400, 8),
    ("5300", "Engineering", "Debit", 900, 8),
    ("5400", "Kitchen", "Debit", 500, 8),
    ("6100", "Sales & Marketing", "Debit", 1500, 5),
    ("6200", "IT", "Debit", 800, 3),
    ("6300", "Security", "Debit", 500, 2),
    ("6400", "Administration", "Debit", 600, 4),
    ("6500", "Front Office", "Debit", 300, 3),
    ("7100", "Finance", "Debit", 700, 2),
]

PROC_DEPTS = {
    "F&B": (35, [("Dairy", 13), ("Meat", 45), ("Produce", 8), ("Beverages", 20), ("Dry goods", 10)]),
    "Kitchen": (15, [("Seafood", 60), ("Bakery", 9), ("Spices", 25)]),
    "Housekeeping": (20, [("Linen", 55), ("Toiletries", 6), ("Cleaning supplies", 15)]),
    "Engineering": (12, [("Spare parts", 120), ("Lighting", 30)]),
    "Spa & Wellness": (6, [("Spa products", 40)]),
    "IT": (4, [("Network equipment", 300)]),
    "Administration": (8, [("Stationery", 5)]),
}

POS_REVIEW = [
    "Great room and a lovely view.", "Spotless room and a very comfortable bed.",
    "Staff were friendly and helpful throughout.", "Breakfast had plenty of choice.",
    "Wi-Fi was fast in the room.", "Check-in was quick and easy.",
    "The pool area was relaxing.", "Housekeeping did an excellent job.",
]
NEG_REVIEW = [
    "Check-in took far too long.", "The room was noisy at night.",
    "Wi-Fi kept dropping out.", "The room was not cleaned until late afternoon.",
    "Restaurant service was slow.", "The air conditioning was too loud.",
    "Breakfast ran out of options early.", "Room service arrived cold.",
]
NEU_REVIEW = [
    "Decent stay overall.", "The room was fine but nothing special.",
    "Good location, average facilities.",
]


def wchoice(rng, pairs):
    items, weights = zip(*pairs)
    return rng.choices(items, weights=weights, k=1)[0]


def n_for_day(rng, base, scale):
    mean = base * scale
    return max(1, int(round(rng.gauss(mean, mean * 0.05))))


def dept_name(rng, canonical, p=0.25):
    if canonical in DEPT_VARIANTS and rng.random() < p:
        return rng.choice(DEPT_VARIANTS[canonical])
    return canonical


def short_room(rng, num):
    return str(num) if rng.random() < 0.92 else f"{num:04d}"


def blank_one(rng, row, nullable, p, empty=""):
    if nullable and rng.random() < p:
        row[rng.choice(nullable)] = empty


def add_duplicates(rng, rows, p):
    dups = [dict(r) for r in rows if rng.random() < p]
    return rows + dups


def write_csv(path, header, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        w.writerows(rows)


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def ymd(d):
    return d.strftime("%Y%m%d")


def main():
    ap = argparse.ArgumentParser(description="Synthetic hotel data generator")
    ap.add_argument("--start", default="2026-10-01", help="first day (YYYY-MM-DD)")
    ap.add_argument("--days", type=int, default=31)
    ap.add_argument("--warmup", type=int, default=7, help="pre-start days simulated so the hotel is already occupied on day 1 (no files written)")
    ap.add_argument("--scale", type=float, default=1.0, help="volume multiplier (0.1 = 10%%)")
    ap.add_argument("--seed", type=int, default=312)
    ap.add_argument("--out", default="data")
    ap.add_argument("--missing", type=float, default=0.02, help="missing-value rate")
    ap.add_argument("--dup", type=float, default=0.01, help="duplicate rate (POS, Wi-Fi)")
    a = ap.parse_args()

    rng = random.Random(a.seed)
    start = datetime.strptime(a.start, "%Y-%m-%d").date()
    days = [start + timedelta(days=i) for i in range(a.days)]
    last_day = days[-1]
    for sub in ("pms", "pos", "procurement", "erp", "wifi", "reviews", "hr"):
        os.makedirs(os.path.join(a.out, sub), exist_ok=True)

    floors = max(2, round(15 * a.scale))
    all_rooms = [f * 100 + n for f in range(1, floors + 1) for n in range(1, 51)]
    free = list(all_rooms)
    releases = defaultdict(list)  # date -> rooms freed that morning

    reservations = []
    res_counter = 104000
    gen_days = [start - timedelta(days=a.warmup) + timedelta(days=i)
                for i in range(a.warmup + a.days + 1)]   # warm-up ... last day + 1
    for d in gen_days:
        free.extend(releases.pop(d, []))
        for _ in range(n_for_day(rng, DAILY_ROWS["arrivals"], a.scale)):
            res_counter += 1
            nights = wchoice(rng, STAY_NIGHTS)
            rtype, base_rate, _w = rng.choices(ROOM_TYPES, weights=[t[2] for t in ROOM_TYPES])[0]
            channel = wchoice(rng, CHANNELS)
            rate = base_rate * rng.uniform(0.85, 1.25) * (0.9 if channel == "Corporate" else 1.0)
            u = rng.random()
            cancelled = u < 0.06
            no_show = 0.06 <= u < 0.08
            active = not (cancelled or no_show)
            if active:
                if not free:          # hotel full -> reservation not made
                    res_counter -= 1
                    continue
                room = free.pop(rng.randrange(len(free)))
                releases[d + timedelta(days=nights)].append(room)
            else:
                room = rng.choice(all_rooms)   # never occupies the room
            co = d + timedelta(days=nights)
            kind = "Cancelled" if cancelled else "No show" if no_show else "Active"
            reservations.append({
                "id": f"RES-{res_counter}", "guest": f"G-{rng.randint(10000, 69999)}",
                "room": room, "rtype": rtype, "ci": d, "co": co,
                "rate": round(rate, 2), "channel": channel, "kind": kind,
                "active": active, "devices": None,
            })

    in_house = defaultdict(list)
    day_set = set(days)
    for r in reservations:
        if not r["active"]:
            continue
        for k in range((r["co"] - r["ci"]).days):
            dd = r["ci"] + timedelta(days=k)
            if dd in day_set:
                in_house[dd].append(r)

    reviews_by_day = defaultdict(list)
    rev_counter = 3000
    for r in reservations:
        if not r["active"] or r["co"] > last_day or rng.random() > 0.27:
            continue
        rev_date = r["co"] + timedelta(days=rng.randint(0, 5))
        if rev_date not in day_set:
            continue
        rev_counter += 1
        score = int(min(10, max(1, round(rng.gauss(7.6, 1.9)))))
        if score >= 8:
            parts = rng.sample(POS_REVIEW, 2) + (rng.sample(NEG_REVIEW, 1) if rng.random() < 0.25 else [])
        elif score >= 5:
            parts = rng.sample(POS_REVIEW, 1) + rng.sample(NEG_REVIEW, 1) + (rng.sample(NEU_REVIEW, 1) if rng.random() < 0.4 else [])
        else:
            parts = rng.sample(NEG_REVIEW, 2) + (rng.sample(POS_REVIEW, 1) if rng.random() < 0.3 else [])
        rng.shuffle(parts)
        row = {
            "review_id": f"REV-{rev_counter}", "reservation_id": r["id"],
            "review_date": rev_date.isoformat(), "score": score,
            "review_text": " ".join(parts),
        }
        blank_one(rng, row, ["reservation_id", "score", "review_text"], a.missing, None)
        reviews_by_day[rev_date].append(row)

    pos_counter, po_counter, gl_counter, sess_counter = 88000, 20900, 773000, 0
    device_counter = 0
    totals = defaultdict(int)

    for d in days:
        iso = d.isoformat()
        occupied = in_house.get(d, [])
        occupancy = len(occupied) / len(all_rooms)

        pms = []
        for r in reservations:
            if not r["active"]:
                status = r["kind"] if r["ci"] == d else None
            elif r["ci"] <= d < r["co"]:
                status = "In house"
            elif r["co"] == d:
                status = "Checked out"
            elif r["ci"] == d + timedelta(days=1):
                status = "Confirmed"
            else:
                status = None
            if status is None:
                continue
            row = {
                "reservation_id": r["id"], "guest_id": r["guest"],
                "room_number": f"{r['room']:04d}", "room_type": r["rtype"],
                "check_in_date": r["ci"].strftime("%d/%m/%Y"),
                "check_out_date": r["co"].strftime("%d/%m/%Y"),
                "rate_per_night": f"{r['rate']:.2f}", "booking_channel": r["channel"],
                "status": status,
            }
            blank_one(rng, row, ["guest_id", "room_number", "room_type", "rate_per_night",
                                 "booking_channel", "status"], a.missing)
            pms.append(row)
        write_csv(os.path.join(a.out, "pms", f"pms_reservations_{ymd(d)}.csv"),
                  ["reservation_id", "guest_id", "room_number", "room_type", "check_in_date",
                   "check_out_date", "rate_per_night", "booking_channel", "status"], pms)

        pos = []
        for _ in range(n_for_day(rng, DAILY_ROWS["pos"], a.scale)):
            pos_counter += 1
            outlet = wchoice(rng, [("Restaurant", 50), ("Bar", 20), ("Room service", 30)])
            if outlet == "Restaurant":
                hour = rng.choice([7, 8, 9, 10, 12, 13, 14, 19, 20, 21, 22])
                pay = wchoice(rng, [("Room charge", 30), ("Card", 55), ("Cash", 15)])
            elif outlet == "Bar":
                hour = rng.choice([17, 18, 19, 20, 21, 22, 23])
                pay = wchoice(rng, [("Room charge", 25), ("Card", 50), ("Cash", 25)])
            else:
                hour = rng.randint(6, 23)
                pay = wchoice(rng, [("Room charge", 75), ("Card", 25)])
            if pay == "Room charge" and not occupied:
                pay = "Card"
            covers = 1 if outlet == "Room service" and rng.random() < 0.6 else wchoice(
                rng, [(1, 20), (2, 40), (3, 15), (4, 15), (5, 5), (6, 5)])
            ts = datetime(d.year, d.month, d.day, hour, rng.randint(0, 59), 0)
            room = short_room(rng, rng.choice(occupied)["room"]) if pay == "Room charge" else ""
            row = {
                "check_id": f"CHK-{pos_counter}", "outlet": outlet,
                "datetime": ts.isoformat(), "total_amount": f"{covers * rng.uniform(70, 230):.2f}",
                "payment_method": pay, "room_number": room, "covers": covers,
            }
            nullable = ["outlet", "datetime", "total_amount", "payment_method", "covers"]
            if pay == "Room charge":
                nullable.append("room_number")
            blank_one(rng, row, nullable, a.missing)
            pos.append(row)
        pos = add_duplicates(rng, pos, a.dup)
        write_csv(os.path.join(a.out, "pos", f"pos_checks_{ymd(d)}.csv"),
                  ["check_id", "outlet", "datetime", "total_amount", "payment_method",
                   "room_number", "covers"], pos)

        proc = []
        proc_depts = [(k, v[0]) for k, v in PROC_DEPTS.items()]
        for _ in range(n_for_day(rng, DAILY_ROWS["proc"], a.scale)):
            po_counter += 1
            dept = wchoice(rng, proc_depts)
            cat, med = rng.choice(PROC_DEPTS[dept][1])
            row = {
                "po_id": f"PO-{po_counter}", "supplier_id": f"SUP-{rng.randint(100, 260)}",
                "department": dept_name(rng, dept), "item_category": cat,
                "quantity": rng.randint(5, 200),
                "unit_cost": f"{med * math.exp(rng.gauss(0, 0.25)):.2f}",
                "order_date": iso,
                "received_date": (d + timedelta(days=rng.randint(1, 5))).isoformat(),
            }
            blank_one(rng, row, ["supplier_id", "department", "item_category", "quantity",
                                 "unit_cost", "received_date"], a.missing)
            proc.append(row)
        write_csv(os.path.join(a.out, "procurement", f"procurement_po_{ymd(d)}.csv"),
                  ["po_id", "supplier_id", "department", "item_category", "quantity",
                   "unit_cost", "order_date", "received_date"], proc)

        erp = []
        for _ in range(n_for_day(rng, DAILY_ROWS["erp"], a.scale)):
            gl_counter += 1
            code, dept, etype, med, _w = rng.choices(ERP_ACCOUNTS, weights=[x[4] for x in ERP_ACCOUNTS])[0]
            if rng.random() < 0.04:
                etype = "Debit" if etype == "Credit" else "Credit"
            row = {
                "entry_id": f"GL-{gl_counter}", "date": iso, "account_code": code,
                "department": dept_name(rng, dept),
                "amount": f"{med * math.exp(rng.gauss(0, 0.6)):.2f}", "entry_type": etype,
            }
            blank_one(rng, row, ["account_code", "department", "amount", "entry_type"], a.missing)
            erp.append(row)
        write_csv(os.path.join(a.out, "erp", f"erp_entries_{ymd(d)}.csv"),
                  ["entry_id", "date", "account_code", "department", "amount", "entry_type"], erp)

        wifi = []
        if occupied:
            for _ in range(n_for_day(rng, DAILY_ROWS["wifi"], a.scale)):
                sess_counter += 1
                st = rng.choice(occupied)
                if st["devices"] is None:
                    st["devices"] = []
                    for _k in range(rng.randint(1, 3)):
                        device_counter += 1
                        st["devices"].append(hashlib.sha256(f"{a.seed}-dev-{device_counter}".encode()).hexdigest()[:16])
                login = datetime(d.year, d.month, d.day) + timedelta(seconds=rng.randint(0, 86399 - 120))
                dur = rng.choice([5, 10, 20, 30, 45, 60, 90, 120, 180, 300]) * rng.uniform(0.6, 1.4)
                logout = min(login + timedelta(minutes=dur), datetime(d.year, d.month, d.day, 23, 59, 59))
                mins = (logout - login).total_seconds() / 60
                row = {
                    "session_id": f"WS-{hashlib.md5(str(sess_counter).encode()).hexdigest()[:5]}",
                    "device_id": rng.choice(st["devices"]),
                    "room_number": short_room(rng, st["room"]),
                    "login_time": login.isoformat(timespec="seconds"),
                    "logout_time": logout.isoformat(timespec="seconds"),
                    "data_used_mb": round(mins * math.exp(rng.gauss(1.5, 0.8)), 1),
                }
                blank_one(rng, row, ["device_id", "room_number", "logout_time", "data_used_mb"],
                          a.missing, None)
                wifi.append(row)
            wifi = add_duplicates(rng, wifi, a.dup)
        write_jsonl(os.path.join(a.out, "wifi", f"wifi_sessions_{ymd(d)}.json"), wifi)

        write_jsonl(os.path.join(a.out, "reviews", f"reviews_{ymd(d)}.json"), reviews_by_day.get(d, []))

        hr = []
        for dept in DEPARTMENTS:
            factor = 0.5 + 0.5 * (occupancy / 0.75)
            staff = max(1, int(round(HR_BASE[dept] * a.scale * factor * rng.uniform(0.92, 1.08))))
            row = {
                "date": iso, "department": dept_name(rng, dept), "staff_on_shift": staff,
                "total_hours": f"{staff * rng.uniform(7.5, 8.8):.1f}",
            }
            blank_one(rng, row, ["department", "staff_on_shift", "total_hours"], a.missing)
            hr.append(row)
        write_csv(os.path.join(a.out, "hr", f"hr_staffing_{ymd(d)}.csv"),
                  ["date", "department", "staff_on_shift", "total_hours"], hr)

        for k, v in (("pms", pms), ("pos", pos), ("procurement", proc), ("erp", erp),
                     ("wifi", wifi), ("reviews", reviews_by_day.get(d, [])), ("hr", hr)):
            totals[k] += len(v)

    print(f"Generated {len(days)} days ({days[0]} to {days[-1]}), scale={a.scale}, "
          f"rooms={len(all_rooms)}, avg occupancy={sum(len(in_house[x]) for x in days) / len(days) / len(all_rooms):.0%}, seed={a.seed} -> {os.path.abspath(a.out)}")
    for k, v in totals.items():
        print(f"  {k:12s} {v:>9,d} rows  (~{v // len(days):,}/day)")


if __name__ == "__main__":
    main()


Overwriting generate_data.py


In [23]:
!ls -lh generate_data.py
!head -20 generate_data.py

-rw-r--r-- 1 root root 18K Nov 18 09:49 generate_data.py
import argparse
import csv
import hashlib
import json
import math
import os
import random
from collections import defaultdict
from datetime import date, datetime, timedelta

DAILY_ROWS = {"arrivals": 207, "pos": 1500, "proc": 100, "erp": 2000, "wifi": 5000}

ROOM_TYPES = [
    ("Standard Twin", 550, 25),
    ("Deluxe King", 950, 35),
    ("Deluxe Twin", 850, 15),
    ("Executive Suite", 1800, 15),
    ("Family Room", 1200, 10),
]
CHANNELS = [("Direct", 35), ("OTA", 45), ("Corporate", 20)]


In [24]:
!python generate_data.py --start 2026-10-01 --days 31 --out data

Generated 31 days (2026-10-01 to 2026-10-31), scale=1.0, rooms=750, avg occupancy=71%, seed=312 -> /content/data
  pms             28,802 rows  (~929/day)
  pos             47,754 rows  (~1,540/day)
  procurement      3,078 rows  (~99/day)
  erp             62,565 rows  (~2,018/day)
  wifi           156,684 rows  (~5,054/day)
  reviews          1,620 rows  (~52/day)
  hr                 465 rows  (~15/day)


In [25]:
!ls data
!ls data/pos | head -5
!ls data/pos | wc -l

erp  hr  pms  pos  procurement	reviews  wifi
pos_checks_20261001.csv
pos_checks_20261002.csv
pos_checks_20261003.csv
pos_checks_20261004.csv
pos_checks_20261005.csv
31


In [26]:
!head -3 data/pms/pms_reservations_20261015.csv
!echo
!head -3 data/pos/pos_checks_20261015.csv
!echo
!head -3 data/procurement/procurement_po_20261015.csv
!echo
!head -3 data/erp/erp_entries_20261015.csv
!echo
!head -3 data/hr/hr_staffing_20261015.csv
!echo
!head -2 data/wifi/wifi_sessions_20261015.json
!echo
!head -2 data/reviews/reviews_20261015.json

reservation_id,guest_id,room_number,room_type,check_in_date,check_out_date,rate_per_night,booking_channel,status
RES-106895,G-43383,0730,Executive Suite,08/10/2026,15/10/2026,1658.57,Corporate,Checked out
RES-106918,G-68735,1046,Standard Twin,08/10/2026,15/10/2026,594.92,OTA,Checked out

check_id,outlet,datetime,total_amount,payment_method,room_number,covers
CHK-109186,Room service,2026-10-15T11:52:00,231.10,Room charge,613,2
CHK-109187,Bar,2026-10-15T20:09:00,262.52,Card,,2

po_id,supplier_id,department,item_category,quantity,unit_cost,order_date,received_date
PO-22298,SUP-217,Food and Beverage,Produce,95,5.27,2026-10-15,2026-10-19
PO-22299,SUP-220,Housekeeping,Linen,129,71.10,2026-10-15,2026-10-16

entry_id,date,account_code,department,amount,entry_type
GL-801739,2026-10-15,6400,Administration,454.67,Debit
GL-801740,2026-10-15,5100,F&B,1644.45,Debit

date,department,staff_on_shift,total_hours
2026-10-15,Rooms,30,243.3
2026-10-15,Front Desk,24,208.6

{"session_id": "WS-65627", "device

In [27]:
import csv, glob, collections, json

pms = next(csv.DictReader(open("data/pms/pms_reservations_20261015.csv")))
pos = next(csv.DictReader(open("data/pos/pos_checks_20261015.csv")))
print("PMS date:", pms["check_in_date"], "| POS datetime:", pos["datetime"])

print("PMS room:", pms["room_number"])
rooms = [r["room_number"] for f in glob.glob("data/pos/*.csv") for r in csv.DictReader(open(f)) if r["room_number"]]
print("POS rooms padded to 4 digits:", sum(1 for x in rooms if len(x) == 4 and x.startswith("0")), "of", len(rooms))

rows = [r for f in glob.glob("data/pms/*.csv") for r in csv.DictReader(open(f))]
blanks = sum(1 for r in rows if any(v == "" for v in r.values()))
print("PMS rows with a blank field: %.1f%%" % (100 * blanks / len(rows)))

ids = [r["check_id"] for f in glob.glob("data/pos/*.csv") for r in csv.DictReader(open(f))]
print("POS rows:", len(ids), "| duplicated check_ids:", len(ids) - len(set(ids)))

names = collections.Counter(r["department"] for f in glob.glob("data/erp/*.csv") for r in csv.DictReader(open(f)))
print(sorted(names))

PMS date: 08/10/2026 | POS datetime: 2026-10-15T11:52:00
PMS room: 0730
POS rooms padded to 4 digits: 970 of 20262
PMS rows with a blank field: 2.1%
POS rows: 47754 | duplicated check_ids: 487
['', 'Administration', 'Banquets', 'Eng', 'Engineering', 'F and B', 'F&B', 'FO', 'Finance', 'FnB', 'Food and Beverage', 'Front Desk', 'Front Office', 'HK', 'House Keeping', 'Housekeeping', 'IT', 'Kitchen', 'Maintenance', 'Rooms', 'Rooms Division', 'S&M', 'Sales & Marketing', 'Sales and Marketing', 'Security', 'Spa', 'Spa & Wellness', 'Spa and Wellness', 'housekeeping', 'rooms']


In [28]:
from datetime import datetime
latest = {}
for f in sorted(glob.glob("data/pms/*.csv")):
    for r in csv.DictReader(open(f)):
        latest[r["reservation_id"]] = r
stays = collections.defaultdict(list)
for r in latest.values():
    if r["status"] in ("Cancelled", "No show", "Confirmed") or not r["room_number"]:
        continue
    ci = datetime.strptime(r["check_in_date"], "%d/%m/%Y").date()
    co = datetime.strptime(r["check_out_date"], "%d/%m/%Y").date()
    stays[int(r["room_number"])].append((ci, co))
total = bad = 0
for f in glob.glob("data/pos/*.csv"):
    for r in csv.DictReader(open(f)):
        if r["payment_method"] == "Room charge" and r["room_number"] and r["datetime"]:
            d = datetime.fromisoformat(r["datetime"]).date()
            total += 1
            if sum(ci <= d < co for ci, co in stays[int(r["room_number"])]) != 1:
                bad += 1
print("Unique reservations:", len(latest))
print("Room charges:", total, "| not matching exactly one stay:", bad)

Unique reservations: 7193
Room charges: 20131 | not matching exactly one stay: 54


In [18]:
!zip -qr data.zip data
!ls -lh data.zip
from google.colab import files
files.download('data.zip')

-rw-r--r-- 1 root root 6.4M Nov 18 09:30 data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!du -sk data/*
!du -sb data

3080	data/erp
128	data/hr
2420	data/pms
2924	data/pos
252	data/procurement
376	data/reviews
27620	data/wifi
37170098	data
